# 04- Préparer les données de mon projet

 Une fois que tout marche ici, range les étapes dans le modèle `pipeline_modele.py`.

In [1]:
import pandas as pd

## 0 · Charger mes sources

In [2]:
# Source 1: Je lis mes 3 fichiers bruts CSV de data/raw/ 
purchase_orders = pd.read_csv("../data/raw/purchase_orders.csv")
parts_master = pd.read_csv("../data/raw/parts_master.csv")
supply_chain_history = pd.read_csv("../data/raw/supply_chain_history.csv")

In [3]:
# Source 2: Je lis le fichier Excel du GSCPI 
gscpi = pd.read_excel("../data/raw/gscpi_data.xls")

In [4]:
# Source 3 : Je lis la réponse de l'API Frankfurter 
# pd.read_json reconnaît la colonne "date" et la transforme directement en vraie date
taux = pd.read_json("../data/raw/taux_change_eur_usd.json")

## Recette 7 · Nettoyer

Pour chaque source : le diagnostic, puis les corrections.

### SOURCE 1

In [5]:
# Diagnostic table puchases_orders

print(len(purchase_orders), "lignes")
print(purchase_orders.duplicated().sum(), "doublons")
purchase_orders.isna().sum()

29666 lignes
0 doublons


po_id            0
supplier_id      0
site_id          0
part_id          0
order_date       0
promised_date    0
receipt_date     0
ordered_qty      0
received_qty     0
dtype: int64

In [6]:
# Diagnostic table parts_master

print(len(parts_master), "lignes")
print(parts_master.duplicated().sum(), "doublons")
parts_master.isna().sum()

300 lignes
0 doublons


part_id                  0
part_family              0
criticality_class        0
unit_cost                0
lead_time_days           0
supplier_id_primary      0
supplier_risk_class      0
is_repairable            0
shelf_life_days        274
dtype: int64

In [7]:
# Diagnostic table supply_chain_history

print(len(supply_chain_history), "lignes")
print(supply_chain_history.duplicated().sum(), "doublons")
supply_chain_history.isna().sum()

280800 lignes
0 doublons


date                   0
site_id                0
part_id                0
planned_maintenance    0
consumption_qty        0
on_hand_qty            0
backorder_qty          0
blocked_qty            0
forecast_qty           0
forecast_type          0
forecast_uplift_pct    0
dtype: int64

In [8]:
# Conclusion: Aucun doublon, aucune case vide à corriger dans les 3 fichiers.
# Les 274 cases vides de shelf_life_days (parts_master) sont normales : cette colonne ne concerne que les pièces à durée de vie limitée.

### SOURCE 2

In [9]:
# Diagnostic excel GSCPI 
print(len(gscpi), "lignes")
print(gscpi.duplicated().sum(), "doublons")
print(gscpi.isna().sum())

# Je regarde le début et la fin du fichier
print(gscpi.head(7))
print(gscpi.tail(3))

350 lignes
3 doublons
Date     4
GSCPI    4
dtype: int64
          Date     GSCPI
0          NaN       NaN
1          NaN       NaN
2          NaN       NaN
3          NaN       NaN
4  31-Jan-1998 -1.179090
5  28-Feb-1998 -0.489883
6  31-Mar-1998 -0.071228
            Date     GSCPI
347  31-Aug-2026  1.271031
348  30-Sep-2026  1.361180
349  31-Oct-2026  1.261911


In [10]:
# Règle de validité : une ligne doit avoir une date ET une valeur.
# Cela supprime les 4 lignes vides du haut et les doublons. 
gscpi = gscpi.dropna(subset=["Date", "GSCPI"])

print(len(gscpi), "lignes après nettoyage")
print(gscpi.duplicated().sum(), "doublons")
print(gscpi.isna().sum())
print(gscpi.head(3))
print(gscpi.tail(3))

346 lignes après nettoyage
0 doublons
Date     0
GSCPI    0
dtype: int64
          Date     GSCPI
4  31-Jan-1998 -1.179090
5  28-Feb-1998 -0.489883
6  31-Mar-1998 -0.071228
            Date     GSCPI
347  31-Aug-2026  1.271031
348  30-Sep-2026  1.361180
349  31-Oct-2026  1.261911


### SOURCE 3

In [11]:
# Vérifications des doublons/lignes vides
print(len(taux), "lignes")
print(taux.duplicated().sum(), "doublons")
print(taux.isna().sum())

# "base" est la monnaie de départ et "quote" la monnaie d'arrivée.
# Je regarde quelles monnaies elles contiennent, pour vérifier qu'il n'y en a qu'une seule de chaque
print(taux["base"].unique(), taux["quote"].unique())

1096 lignes
0 doublons
date     0
base     0
quote    0
rate     0
dtype: int64
<ArrowStringArray>
['EUR']
Length: 1, dtype: str <ArrowStringArray>
['USD']
Length: 1, dtype: str


In [12]:
# Conclusion : aucun doublon et aucune case vide à corriger.
# base et quote valent toujours EUR et USD.
# La date a été convertie automatiquement en vraie date au chargement.

## Recette 8 · Homogénéiser

### SOURCE 1

In [13]:
# DIAGNOSTIC 1 : le type de chaque colonne
print("--- purchase_orders ---")
print(purchase_orders.dtypes)
print("--- parts_master ---")
print(parts_master.dtypes)
print("--- supply_chain_history ---")
print(supply_chain_history.dtypes)

--- purchase_orders ---
po_id              str
supplier_id        str
site_id            str
part_id            str
order_date         str
promised_date      str
receipt_date       str
ordered_qty      int64
received_qty     int64
dtype: object
--- parts_master ---
part_id                    str
part_family                str
criticality_class          str
unit_cost              float64
lead_time_days           int64
supplier_id_primary        str
supplier_risk_class        str
is_repairable              str
shelf_life_days        float64
dtype: object
--- supply_chain_history ---
date                       str
site_id                    str
part_id                    str
planned_maintenance       bool
consumption_qty          int64
on_hand_qty              int64
backorder_qty            int64
blocked_qty              int64
forecast_qty             int64
forecast_type              str
forecast_uplift_pct    float64
dtype: object


In [14]:
# Les dates sont du texte. Afin de pour pouvoir calculer des écarts, on va devoir modifier le type.

In [15]:
# DIAGNOSTIC 2 : On vérifie que les noms de colonnes et les unités entre chaque fichier sont cohérents

display(purchase_orders.head())
display(parts_master.head())
display(supply_chain_history.head())

,po_id,supplier_id,site_id,part_id,order_date,promised_date,receipt_date,ordered_qty,received_qty
0,PO000001,SUP004,SITE01,P00001,2022-05-02,2022-05-28,2022-05-29,16,16
1,PO000002,SUP004,SITE01,P00001,2022-07-04,2022-07-31,2022-08-02,5,5
2,PO000003,SUP004,SITE01,P00001,2022-10-17,2022-11-12,2022-11-16,5,5
3,PO000004,SUP004,SITE01,P00001,2023-02-06,2023-03-07,2023-03-09,3,3
4,PO000005,SUP004,SITE01,P00001,2023-04-03,2023-04-28,2023-04-29,5,5


,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
0,P00001,Electrical,B,2323.45,27,SUP004,Low,No,735.0
1,P00002,Cabin,C,670.84,31,SUP028,Low,No,NaN
2,P00003,Avionics,C,219.44,49,SUP040,Medium,Yes,NaN
3,P00004,Electrical,A,12488.80,33,SUP034,Low,No,NaN
4,P00005,Cabin,C,1319.96,63,SUP024,Medium,No,NaN


,date,site_id,part_id,planned_maintenance,consumption_qty,on_hand_qty,backorder_qty,blocked_qty,forecast_qty,forecast_type,forecast_uplift_pct
0,2022-01-03,SITE01,P00001,False,2,11,0,0,2,Baseline,0.000
1,2022-01-10,SITE01,P00001,True,4,14,0,0,3,Adjusted,-0.115
2,2022-01-17,SITE01,P00001,True,2,12,0,0,3,Baseline,0.000
3,2022-01-24,SITE01,P00001,False,0,12,0,0,2,Baseline,0.000
4,2022-01-31,SITE01,P00001,True,0,12,0,0,1,Adjusted,-0.110


In [16]:
# Conclusion: Les noms de colonnes et les unités sont déjà cohérents entre les fichiers donc pas de modification à éffectuer ici.

In [17]:
# Modification du type date

# purchase_orders : 3 colonnes de dates
for colonne in ["order_date", "promised_date", "receipt_date"]:
    purchase_orders[colonne] = pd.to_datetime(purchase_orders[colonne])

# supply_chain_history : 1 colonne de date
supply_chain_history["date"] = pd.to_datetime(supply_chain_history["date"])

# parts_master : aucune colonne de date, rien à transformer

# Je vérifie à nouveau le type de chaque colonne, après modification
print(purchase_orders.dtypes)
print(parts_master.dtypes)
print(supply_chain_history.dtypes)

po_id                       str
supplier_id                 str
site_id                     str
part_id                     str
order_date       datetime64[us]
promised_date    datetime64[us]
receipt_date     datetime64[us]
ordered_qty               int64
received_qty              int64
dtype: object
part_id                    str
part_family                str
criticality_class          str
unit_cost              float64
lead_time_days           int64
supplier_id_primary        str
supplier_risk_class        str
is_repairable              str
shelf_life_days        float64
dtype: object
date                   datetime64[us]
site_id                           str
part_id                           str
planned_maintenance              bool
consumption_qty                 int64
on_hand_qty                     int64
backorder_qty                   int64
blocked_qty                     int64
forecast_qty                    int64
forecast_type                     str
forecast_uplift_pct      

In [18]:
# Conclusion : les dates sont maintenant de type datetime. 

### SOURCE 2

In [19]:
# Diagnostic du type des colonnes avant modification 
print(gscpi.dtypes)

gscpi.head

Date         str
GSCPI    float64
dtype: object


<bound method NDFrame.head of             Date     GSCPI
4    31-Jan-1998 -1.179090
5    28-Feb-1998 -0.489883
6    31-Mar-1998 -0.071228
7    30-Apr-1998 -0.146184
8    31-May-1998 -0.471055
..           ...       ...
345  30-Jun-2026  1.202562
346  31-Jul-2026  0.951130
347  31-Aug-2026  1.271031
348  30-Sep-2026  1.361180
349  31-Oct-2026  1.261911

[346 rows x 2 columns]>

In [20]:
# Je donne aux colonnes des noms en minuscules, comme dans mes autres fichiers
gscpi = gscpi.rename(columns={"Date": "date", "GSCPI": "gscpi"})

# Je transforme les dates en type date.
gscpi["date"] = pd.to_datetime(gscpi["date"], format="%d-%b-%Y")

# Le type après modification : la date doit être de type datetime64
print(gscpi.dtypes)
print(gscpi.head)

date     datetime64[us]
gscpi           float64
dtype: object
<bound method NDFrame.head of           date     gscpi
4   1998-01-31 -1.179090
5   1998-02-28 -0.489883
6   1998-03-31 -0.071228
7   1998-04-30 -0.146184
8   1998-05-31 -0.471055
..         ...       ...
345 2026-06-30  1.202562
346 2026-07-31  0.951130
347 2026-08-31  1.271031
348 2026-09-30  1.361180
349 2026-10-31  1.261911

[346 rows x 2 columns]>


In [21]:
# Je vérifie la période couverte, afin de pouvoir croiser le fichier à mes données source 1 (2022 à 2024)
print("Première date :", gscpi["date"].min())
print("Dernière date :", gscpi["date"].max())
periode = gscpi[(gscpi["date"] >= "2022-01-01") & (gscpi["date"] <= "2024-12-31")]
print(len(periode), "mois entre 2022 et 2024")        # je dois obtenir 36

Première date : 1998-01-31 00:00:00
Dernière date : 2026-10-31 00:00:00
36 mois entre 2022 et 2024


### SOURCE 3

In [22]:
# Je retire base et quote (toujours EUR et USD) et je donne un nom plus clair au taux
taux = taux.drop(columns=["base", "quote"])

# Je renomme la colonne "rate" en "taux_eur_usd", pour plus de clareté 
# Exemple : 1,13 veut dire que 1 euro s'échange contre 1,13 dollar
taux = taux.rename(columns={"rate": "taux_eur_usd"})     

# Je vérifie les types après modifications 
print(taux.dtypes)

# Je vérifie la période couverte et le nombre de jours
print("Première date :", taux["date"].min())             # 2022-01-01
print("Dernière date :", taux["date"].max())             # 2024-12-31
print(len(taux), "jours")                                # 1096

date            datetime64[us]
taux_eur_usd           float64
dtype: object
Première date : 2022-01-01 00:00:00
Dernière date : 2024-12-31 00:00:00
1096 jours


## Recette 9 · Joindre, agréger, contrôler

Joindre chaque commande aux informations de sa pièce afin de pourvoir analyser les retards par criticité et par famille, nécessaire pour prioriser les références critiques.

In [23]:
# Etape 1: Je rattache à chaque commande les informations de sa pièce (famille, criticité, risque fournisseur...).
# La colonne commune est part_id. 
# how="left" : je garde TOUTES les commandes, même si une pièce n'avait pas de fiche
commandes = purchase_orders.merge(parts_master, on="part_id", how="left")

# Je compare le nombre de lignes avant et après : il ne doit pas changer
print(len(purchase_orders), "->", len(commandes), "lignes")

# Je regarde le résultat
commandes.head(3)

29666 -> 29666 lignes


,po_id,supplier_id,site_id,part_id,order_date,promised_date,receipt_date,ordered_qty,received_qty,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
0,PO000001,SUP004,SITE01,P00001,2022-05-02,2022-05-28,2022-05-29,16,16,Electrical,B,2323.45,27,SUP004,Low,No,735.0
1,PO000002,SUP004,SITE01,P00001,2022-07-04,2022-07-31,2022-08-02,5,5,Electrical,B,2323.45,27,SUP004,Low,No,735.0
2,PO000003,SUP004,SITE01,P00001,2022-10-17,2022-11-12,2022-11-16,5,5,Electrical,B,2323.45,27,SUP004,Low,No,735.0


In [28]:
# Etape 2: création d'une clé commune "mois" et "année" dans chaque source, au même format

# Pour les commandes (Source 1): l'année et le mois de la commande (order_date)
commandes["annee"] = commandes["order_date"].dt.year      
commandes["mois"] = commandes["order_date"].dt.month     

# Pour l'historique (Source 1): l'année et le mois de la date de la semaine
supply_chain_history["annee"] = supply_chain_history["date"].dt.year
supply_chain_history["mois"] = supply_chain_history["date"].dt.month

# Pour le GSCPI (Source 2): l'année et le mois de la date (qui est la fin du mois)
gscpi["annee"] = gscpi["date"].dt.year
gscpi["mois"] = gscpi["date"].dt.month

# Pour les taux de change (Source 3) : l'année et le mois de chaque jour
taux["annee"] = taux["date"].dt.year
taux["mois"] = taux["date"].dt.month
# L'API donne un taux par jour. Mon GSCPI est mensuel, donc je ramène les taux au mois.
# groupby = je regroupe les jours qui ont la même année et le même mois
# mean = je calcule la moyenne du taux pour chaque groupe
taux_mensuel = taux.groupby(["annee", "mois"], as_index=False)["taux_eur_usd"].mean().round(4)
# Je renomme la colonne pour plus de clarté 
taux_mensuel = taux_mensuel.rename(columns={"taux_eur_usd": "taux_eur_usd_moyen"})

In [29]:
# ÉTAPE 3 : contrôler mes  tables avant de les enregistrer

# Les 36 mois (année + mois) de mes commandes, pour vérifier le croisement avec le GSCPI et les taux
mois_commandes = commandes[["annee", "mois"]].drop_duplicates()

controles = {
    # --- commandes ---
    "commandes : volume": len(commandes) == len(purchase_orders),                                        # la jointure n'a rien perdu ni ajouté
    "commandes : unicité (po_id)": commandes.duplicated(subset=["po_id"]).sum() == 0,                    # une seule ligne par commande
    "commandes : aucune case vide dans criticality_class": commandes["criticality_class"].isna().sum() == 0,   # toutes les commandes ont retrouvé leur pièce
    "commandes : cohérence : réception après commande": (commandes["receipt_date"] >= commandes["order_date"]).all(),

    # --- parts_master (le référentiel des pièces) ---
    "parts_master : volume": len(parts_master) == 300,                                                   # 300 pièces
    "parts_master : unicité (part_id)": parts_master.duplicated(subset=["part_id"]).sum() == 0,          # une seule ligne par pièce

    # --- historique hebdomadaire ---
    "historique : volume": len(supply_chain_history) == 280800,
    "historique : unicité": supply_chain_history.duplicated(subset=["site_id", "part_id", "date"]).sum() == 0,   # une ligne par site, pièce et semaine
    "historique : cohérence : quantités >= 0": (supply_chain_history[["consumption_qty", "on_hand_qty", "backorder_qty", "blocked_qty"]] >= 0).all().all(),

    # --- gscpi et taux de change ---
    "gscpi : unicité (annee + mois)": gscpi.duplicated(subset=["annee", "mois"]).sum() == 0,             # un seul chiffre par mois
    "taux : volume": len(taux_mensuel) == 36,                                                        # un taux par mois, pendant 3 ans
    "taux : unicité (annee + mois)": taux_mensuel.duplicated(subset=["annee", "mois"]).sum() == 0,   # un seul taux par mois
    "croisement : les 36 mois des commandes existent dans taux": len(mois_commandes.merge(taux_mensuel[["annee", "mois"]], on=["annee", "mois"])) == 36,

    # --- croisement : mes données ont-elles une valeur de GSCPI et de taux pour chaque mois ? ---
    "croisement : les 36 mois des commandes existent dans le gscpi": len(mois_commandes.merge(gscpi[["annee", "mois"]], on=["annee", "mois"])) == 36,
    "croisement : les 36 mois des commandes existent dans taux": len(mois_commandes.merge(taux[["annee", "mois"]].drop_duplicates(), on=["annee", "mois"])) == 36,
}
for nom, resultat in controles.items():
    print("OK    " if resultat else "ÉCHEC ", nom)

OK     commandes : volume
OK     commandes : unicité (po_id)
OK     commandes : aucune case vide dans criticality_class
OK     commandes : cohérence : réception après commande
OK     parts_master : volume
OK     parts_master : unicité (part_id)
OK     historique : volume
OK     historique : unicité
OK     historique : cohérence : quantités >= 0
OK     gscpi : unicité (annee + mois)
OK     taux : volume
OK     taux : unicité (annee + mois)
OK     croisement : les 36 mois des commandes existent dans taux
OK     croisement : les 36 mois des commandes existent dans le gscpi


In [30]:
# Résumé : quantités commandées et reçues, additionnées par mois.
resume = commandes.groupby(["annee", "mois"], as_index=False)[["ordered_qty", "received_qty"]].sum()

# Je vérifie que le résumé n'a rien perdu : le total doit être le même qu'avant
print(resume.shape)                                                                   # 36 lignes, 4 colonnes
print(resume["ordered_qty"].sum() == commandes["ordered_qty"].sum())                  # True

# J'enregistre dans data/processed/ (data/raw/ n'est jamais touché)
import os

commandes.to_csv("../data/processed/commandes_enrichies.csv", index=False)
parts_master.to_csv("../data/processed/parts_master.csv", index=False)
supply_chain_history.to_csv("../data/processed/historique_hebdo.csv", index=False)
gscpi.to_csv("../data/processed/gscpi_mensuel.csv", index=False)
resume.to_csv("../data/processed/resume_commandes_par_mois.csv", index=False)
taux_mensuel.to_csv("../data/processed/taux_change_mensuel.csv", index=False)

print("Fichiers enregistrés :", os.listdir("../data/processed"))

(36, 4)
True
Fichiers enregistrés : ['.gitkeep', 'commandes_enrichies.csv', 'gscpi_mensuel.csv', 'historique_hebdo.csv', 'parts_master.csv', 'resume_commandes_par_mois.csv', 'taux_change_jour.csv', 'taux_change_mensuel.csv']
